<a href="https://colab.research.google.com/github/udayRage/cuPAMI/blob/main/notebooks/georeferencedFrequentPattern/cuSpatialEclat.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding spatial frequent pattern using cuSpatialEclat

This tutorial shows how to prepare a dataset, run cuSpatialEclat, inspect the generated patterns, and save the results.

[Algorithm documentation](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuSpatialEclat.md)

# Prerequisites

1. Installing cuPAMI and CuPy

Select a GPU runtime in Colab. The following cell installs missing packages. The CuPy package shown is for CUDA 12.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

2. Checking the GPU

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


3. Preparing a sample dataset

There is no dedicated spatial data generator. The small sample below is ready to run. Replace both lists with your own data as needed. Transactions contain item labels only. Each neighbour row starts with an item followed by its neighbours, used as listed. The sample contains three transactions and mutually neighbouring items.

In [3]:
from pathlib import Path
from cupami import cuSpatialEclat

transaction_rows = [
    "a,b,c",
    "a,b",
    "a,c",
]
neighbour_rows = [
    "a,b,c",
    "b,a,c",
    "c,a,b",
]
if not transaction_rows or not neighbour_rows:
    raise ValueError("Add transaction and neighbour rows above before continuing.")

Path("transactions.csv").write_text("\n".join(transaction_rows) + "\n", encoding="utf-8")
Path("neighbours.csv").write_text("\n".join(neighbour_rows) + "\n", encoding="utf-8")


18

## Finding patterns using cuSpatialEclat

### Step 1: Choosing the input parameters

`minSup=0.1` means 10% of transactions. Patterns must also satisfy the neighbour constraints.

### Step 2: Executing cuSpatialEclat

In [4]:
miner = cuSpatialEclat("transactions.csv", "neighbours.csv", minSup=0.1)
miner.mine()


### Step 3: Investigating the generated patterns

Display the number of patterns and the first ten results.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 7
a 3
b 2
c 2
a	b 2
a	c 2
b	c 1
a	b	c 1


### Step 4: Saving patterns and measuring runtime

Save the patterns to `patterns.txt`. Loading, mining, and total runtime are reported in seconds.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 0.00042635994032025337
Mining: 0.17110662907361984
Total: 0.1715329890139401
